# Clase 14: Casos de uso reales — cierre del curso

**Objetivos de aprendizaje:**
- Reconocer aplicaciones reales de la ciencia de datos en negocio, salud y medioambiente.
- Recorrer un flujo de trabajo completo (datos → limpieza → EDA → modelo → informe) sobre un caso real.
- Integrar lo aprendido en las 13 clases anteriores.

**Duración estimada:** 120 min

Esta es la última clase del curso. No hay conceptos nuevos que memorizar — hay una invitación a ver todo lo que ya sabes funcionando junto, de principio a fin.

## La ciencia de datos está en todas partes

- **Uber**: datos en tiempo real para asignar conductores y fijar precios dinámicos.
- **Toronto Transportation Commission**: combinando datos de tranvías, quejas y tráfico, redujo la congestión mensual de 4,75 h (2010) a 3 h (2014).
- **Monitoreo ambiental**: barcos robóticos, boyas y drones para vigilar floraciones de cianobacterias.
- **Recomendación**: Amazon, Netflix, Spotify, Siri, wearables (Fitbit, Apple Watch).
- **UPS (2013)**: optimización de rutas con datos de conductores, vehículos y clientes.
- **Salud**: analítica predictiva para recomendar pruebas/tratamientos; NorthShore University HealthSystem, primer proveedor de EE. UU. con despliegue completo de Historiales Médicos Electrónicos.
- **Predicción de desastres**: Universidad de Warwick combina redes sociales y estaciones meteorológicas para rastrear inundaciones/huracanes.

**Caso Netflix / House of Cards**: analizaron datos de visionado, pausas, rebobinados, búsquedas de actor/director y el éxito de la versión británica original, para dar luz verde al remake antes de rodar un solo episodio. Netflix sabía lo que la gente quería antes de que ella misma lo supiera.

## El viaje de Lila: el caso integrador

Lila, con formación en economía, consigue su primer puesto como data scientist junior en retail. Su tarea: encontrar patrones y anomalías en datos de clientes. Su proceso:

1. **Selección y obtención del dataset** de múltiples fuentes.
2. **Comprensión y limpieza** con Python y SQL: nulos, duplicados, outliers.
3. **EDA**: estadísticas resumen, histogramas, diagramas de dispersión.
4. **Feature engineering**: por ejemplo, calcular el importe total de compra.
5. **Análisis estadístico y ML**: regresión precio-ventas, previsión de demanda, segmentación.
6. **Presentación**: notebook → informe + presentación para el área de eCommerce.
7. **Aprendizaje continuo**: Lila sigue formándose después de este proyecto.

Este es exactamente el proceso de seis pasos del capítulo 1 (clarificar, recolectar, analizar, reconocer patrones, contar la historia, visualizar), aplicado a un caso real. Vamos a recorrerlo nosotros mismos con código, de principio a fin.

## El flujo de trabajo completo, con código

Como sustituto reproducible y 100% local de un caso real de regresión (como el de predicción de área quemada por incendios forestales), usamos el dataset de diabetes integrado en scikit-learn: mismo tipo de problema (predecir un número a partir de variables numéricas), sin descargar nada externo.

### Paso 1 — Cargar los datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

# load_diabetes() viene integrado en scikit-learn: nada de descargas externas.
# Cada fila es un paciente; cada columna, una variable clínica ya normalizada.
# El objetivo ("target") es una medida numérica de progresión de la diabetes al año.
datos = load_diabetes(as_frame=True)
df = datos.frame

print("Filas, columnas:", df.shape)
df.head()

### Paso 2 — Análisis Exploratorio de Datos (EDA)

In [ ]:
# Estadísticas resumen: media, desviación, mínimos y máximos de cada columna
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histograma: cómo se distribuye la variable objetivo
axes[0].hist(df["target"], bins=20, color="steelblue")
axes[0].set_title("Distribución del objetivo (progresión de la enfermedad)")
axes[0].set_xlabel("target")

# Diagrama de dispersión: ¿tiene relación el índice de masa corporal con el objetivo?
axes[1].scatter(df["bmi"], df["target"], alpha=0.5, color="seagreen")
axes[1].set_title("Índice de masa corporal (bmi) vs. objetivo")
axes[1].set_xlabel("bmi (normalizado)")
axes[1].set_ylabel("target")

plt.tight_layout()
plt.show()

El histograma nos dice si la variable objetivo está sesgada (si lo estuviera mucho, como en el caso real de incendios forestales, convendría aplicar una transformación logarítmica antes de modelar). El diagrama de dispersión nos da una primera pista visual: parece existir una relación positiva entre `bmi` y `target` — a mayor índice de masa corporal, mayor progresión de la enfermedad.

### Paso 3 — Limpieza mínima (simulando un problema real de datos)

El dataset ya viene limpio, así que —como hizo Lila con datos reales de clientes— vamos a inyectar nosotros mismos un problema de datos faltantes para practicar cómo se trata.

In [ ]:
# Simulamos que un 5% de los valores de "bmi" se perdieron (por ejemplo,
# un sensor que falló al registrar el dato de algunos pacientes).
n_filas = len(df)
indices_nulos = np.random.choice(df.index, size=int(n_filas * 0.05), replace=False)
df.loc[indices_nulos, "bmi"] = np.nan

print("Valores nulos antes de limpiar:", df["bmi"].isna().sum())

# Imputamos con la mediana: una técnica sencilla y razonable cuando faltan pocos valores
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Valores nulos después de limpiar:", df["bmi"].isna().sum())

### Paso 4 — Ingeniería de características (feature engineering)

In [ ]:
# Creamos una columna derivada sencilla: un "índice combinado" de riesgo metabólico
# que suma bmi y presión arterial media (bp). No es una fórmula médica real —
# es un ejemplo de cómo se construye una característica nueva a partir de otras,
# igual que Lila calculó el importe total de compra a partir de columnas más básicas.
df["indice_riesgo"] = df["bmi"] + df["bp"]

df[["bmi", "bp", "indice_riesgo"]].head()

### Paso 5 — Entrenar y evaluar un modelo de regresión

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

mse = mean_squared_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"Error cuadrático medio (MSE): {mse:.2f}")
print(f"R² (proporción de varianza explicada): {r2:.2f}")

El `mean_squared_error` mide, en promedio, cuánto se equivoca el modelo (cuanto más bajo, mejor). El `r2_score` indica qué proporción de la variación del objetivo explica el modelo: 1.0 sería una predicción perfecta, 0.0 equivaldría a no explicar nada mejor que la media.

Podemos comparar este modelo lineal con un árbol de decisión, para ver si captura mejor relaciones no lineales entre las variables:

In [ ]:
arbol = DecisionTreeRegressor(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)
predicciones_arbol = arbol.predict(X_test)

mse_arbol = mean_squared_error(y_test, predicciones_arbol)
r2_arbol = r2_score(y_test, predicciones_arbol)

print(f"Árbol de decisión — MSE: {mse_arbol:.2f}, R²: {r2_arbol:.2f}")
print(f"Regresión lineal  — MSE: {mse:.2f}, R²: {r2:.2f}")

### Paso 6 — Informe final

Sobre este conjunto de datos de diabetes, el análisis muestra una relación clara y positiva entre el índice de masa corporal y la progresión de la enfermedad al año siguiente, visible tanto en el diagrama de dispersión como en el peso que le asigna el modelo de regresión lineal. El modelo entrenado explica una parte relevante pero no total de la variación del objetivo, lo cual es esperable con modelos simples y un conjunto reducido de variables clínicas. Antes de usar este modelo en un contexto real haría falta seguir explorando modelos y variables adicionales, pero el flujo de trabajo aplicado aquí —desde la carga de datos hasta la evaluación— es el mismo que usarías en cualquier problema de regresión con datos tabulares, sea prediciendo progresión de una enfermedad, precios de vivienda o área quemada por un incendio forestal.

## Resumen de la clase y cierre del curso

- La ciencia de datos ya está detrás de decisiones cotidianas: Uber, Netflix, UPS, gestión de tráfico, salud, predicción de desastres.
- El caso de Lila resume, en un contexto real, el mismo proceso de seis pasos del capítulo 1.
- Un flujo de trabajo técnico completo: cargar → EDA → limpiar → feature engineering → modelar → evaluar → informe.
- Ningún modelo es perfecto a la primera — el valor está en el proceso completo y en comunicar honestamente los resultados (clase 13).

Con esta clase se cierra el recorrido completo del curso: desde "¿qué es la ciencia de datos?" (clase 1) hasta un flujo de trabajo real de principio a fin. El resto es práctica constante.

**Manual:** [16 — Casos de uso reales](../manual/16_casos_de_uso_reales.md)

**Práctica (capstone):** [Clase 14 — Práctica](../notebooks_practica/clase14_practica_casos_de_uso_reales.ipynb)